In [1]:
#LIBARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
!pip install xgboost

In [3]:
#
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

In [4]:
#
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [5]:
train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [6]:
x_train = train.drop(columns=["id","addicted_label"])
y_train = train["addicted_label"]

x_test = test.drop(columns=["id"])

In [7]:
categoric_cols = ['gender', 'stress_level', 'academic_work_impact']

for col in categoric_cols:
    x_train[col] = x_train[col].astype('category')
    x_test[col] = x_test[col].astype('category')

In [8]:
# 2. Re-create stratified train/validation split
x_trn, x_val, y_trn, y_val = train_test_split(
    x_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

In [10]:
# 3. Instantiate XGBoost Classifier
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',          # Histogram method for high performance on large datasets
    enable_categorical=True,      # Native support for pandas category dtypes
    early_stopping_rounds=50,     # Early stopping to prevent overfitting
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

# 4. Fit model with early stopping on validation performance
xgb_model.fit(
    x_trn, y_trn,
    eval_set=[(x_val, y_val)],
    verbose=False
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,'auc'


In [11]:
xgb_val_probs = xgb_model.predict_proba(x_val)[:, 1]
print("XGBoost Best Iteration:", xgb_model.best_iteration)
print("XGBoost Val ROC AUC:    ", roc_auc_score(y_val, xgb_val_probs))
print("XGBoost Val Log Loss:   ", log_loss(y_val, xgb_val_probs))

XGBoost Best Iteration: 999
XGBoost Val ROC AUC:     0.9602786357950663
XGBoost Val Log Loss:    0.23204675478015466


In [12]:
# 1. Instantiate final model using optimal number of trees (best_iteration + 1)
final_xgb = xgb.XGBClassifier(
    n_estimators=xgb_model.best_iteration + 1,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

# 2. Fit on 100% full training dataset
final_xgb.fit(x_train, y_train)

# 3. Predict probabilities on x_test
xgb_test_probability = final_xgb.predict_proba(x_test)[:, 1]

In [13]:
xgb_output_df = pd.DataFrame({
    'id': test['id'],
    'predicted_probability': xgb_test_probability
})


xgb_output_df.head(10)

,id,predicted_probability
0,691369,0.998843
1,691370,0.908651
2,691371,0.970562
3,691372,0.987751
4,691373,0.996298
5,691374,0.819905
6,691375,0.851819
7,691376,0.370197
8,691377,0.975234
9,691378,0.422681


In [14]:
xgb_output_df.to_csv('submission.csv', index=False)